# GroupBy

"How many complaints per borough?" "Which weekday is busiest?" Questions like these all have the same shape: *do something for each group*. In pandas, `groupby` answers them in one line.

**Learning outcomes**

- Explain **split-apply-combine** in your own words.
- Count and summarise per group with `groupby(...).size()` and `.agg(...)` using named aggregation.
- Group by several keys, sort the result and turn it back into a normal table with `reset_index()`.
- Use the "mean of a True/False column" trick to get a share (percentage) per group.
- Group a time series by weekday or month and plot the result with a title and labels.

**Before you start:** you should know how to read a CSV (`08_ReadingCSV`), filter rows (`10_Filter`), and parse dates, handle missing values and use `.dt` (`11_Dates_and_Missing_Values`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from course_checks import check, check_equal
from course_viz import show_groupby, explore_groupby, legend
from course_viz.grouping import (groupby_steps, compare_three, styled_agg, bar_labeled,
                                 weekday_bars, month_line, weekday_month_heatmap, pivot_heatmap)

First, the NYC 311 data again, prepared exactly as in the last notebook: dates parsed, impossible durations turned into `NaN`, and `'Unspecified'` boroughs turned into missing values.

In [ ]:
df = pd.read_csv('311-service-requests.csv', dtype={'Incident Zip': str})

FMT = '%m/%d/%Y %I:%M:%S %p'
df['Created'] = pd.to_datetime(df['Created Date'], format=FMT)
df['Closed'] = pd.to_datetime(df['Closed Date'], format=FMT)

df['hours'] = (df['Closed'] - df['Created']).dt.total_seconds() / 3600
df.loc[df['hours'] < 0, 'hours'] = np.nan

df['Borough'] = df['Borough'].replace('Unspecified', np.nan)
df.shape

**You should see:** `(111069, 55)`: our 52 original columns plus `Created`, `Closed` and `hours`.

## 1. The idea: split, apply, combine

Imagine a pile of index cards, one per complaint, each with a borough written on it. To count complaints per borough you would:

1. **Split** the pile into smaller piles, one per borough.
2. **Apply** a calculation to each pile (count the cards, take an average...).
3. **Combine** the answers into one small table.

`groupby` does all three steps for you:

```text
 complaints                SPLIT                APPLY        COMBINE
 ----------           ---------------         ---------     ----------------
 BRONX                 BRONX   | BRONX  ->    count = 2     BRONX       2
 QUEENS                QUEENS  | QUEENS ->    count = 3     QUEENS      3
 BRONX
 QUEENS
 QUEENS
```

In code: `df.groupby('Borough')` (split), then say what to compute (apply), and pandas assembles the result (combine). By itself `groupby(...)` does nothing visible. It just remembers the split. You must follow it with a calculation.

### Watch it happen on real rows

The picture below uses real complaints. **Left:** the rows are split, each borough gets its own colour. **Middle:** one calculation runs inside each colour. **Right:** the answers are combined into a small table that keeps the colours.

In [ ]:
four = df[df['Borough'].isin(['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS'])]
show_groupby(four, 'Borough', 'hours', 'mean')

**What to notice:** every colour in the right-hand table matches a block of rows on the left; four groups in, four rows out. The result has one row per *group*, not per complaint. (Only 3 rows per group are drawn, but the average uses all of them.)

Now the same thing with all the arithmetic spelled out. A tiny made-up table (complaints per day in three boroughs) lets you check every number by hand. Change `'sum'` to `'mean'`, `'max'`, `'median'` or `'size'` and run it again.

In [ ]:
tiny = pd.DataFrame({'Borough': ['QUEENS', 'BRONX', 'QUEENS', 'BRONX', 'BRONX', 'BROOKLYN'],
                     'complaints': [4, 3, 1, 5, 2, 6]})
groupby_steps(tiny, 'Borough', 'complaints', 'sum')

**What to notice:** the rows in stage 1 are in arrival order; stage 2 only *regroups* them, nothing is calculated yet. The calculation (stage 3) happens once per group, and stage 4 is just those answers stacked up.

Try it yourself: choose the key and the calculation and watch split, apply and combine redraw.

In [ ]:
_ = explore_groupby(df, ['Borough', 'Agency', 'Status'], 'hours')

**What to notice:** with `size` the numbers are group sizes; with `mean` or `median` they are typical durations. Changing the *key* changes how the rows are coloured; changing the *calculation* changes only the middle and right panes.

## 2. Counting per group with `size()`

`size()` counts the rows in each group.

In [ ]:
df.groupby('Borough').size()

**You should see:** five boroughs in alphabetical order: `BRONX 19686`, `BROOKLYN 32890`, `MANHATTAN 24288`, `QUEENS 22281`, `STATEN ISLAND 4817`. Rows with a missing borough are skipped by default.

It is the same answer as `df['Borough'].value_counts()`, but `groupby` can do much more. To get the largest first, sort:

In [ ]:
df.groupby('Borough').size().sort_values(ascending=False)

**You should see:** the same numbers, now in descending order: Brooklyn, Manhattan, Queens, Bronx, Staten Island.

### `groupby` vs `value_counts` vs a boolean mean

Three one-liners that look alike. A and B answer the *same* question (how many rows per borough?). C answers a *different* one (what share of each borough is noise?).

In [ ]:
noise = df['Complaint Type'].str.contains('Noise')
compare_three(df, 'Borough', noise, 'noise')

**What to notice:** the bars in A and B are identical, so `value_counts()` is a shortcut for counting only. C has a different unit (percent): Manhattan wins on share even though Brooklyn has more complaints overall. You will use C again in section 5.

### ✏️ Your turn

Which agencies receive the most complaints? Count the rows per `Agency`, largest first, and store the result in `agency_counts`.

In [ ]:
agency_counts = ...  # <- replace ... with your code

In [ ]:
check_equal(lambda: agency_counts.head(3).to_dict(), {'HPD': 39067, 'DOT': 17707, 'NYPD': 15295},
            hint="df.groupby('Agency').size().sort_values(ascending=False)")

<details>
<summary>Show solution</summary>

```python
agency_counts = df.groupby('Agency').size().sort_values(ascending=False)
```

</details>

## 3. Several statistics at once with `.agg`

Choose a column, then a function: `df.groupby('Borough')['hours'].median()`. To compute several different things at once, use **named aggregation**. Each keyword becomes a column of the result, written as `new_name=('column', 'function')`:

In [ ]:
by_borough = df.groupby('Borough').agg(
    complaints=('Unique Key', 'size'),
    median_hours=('hours', 'median'),
    open_or_unknown=('hours', lambda s: s.isna().sum()),
)
by_borough

**You should see:** a table with five rows (the boroughs) and three columns. `complaints` matches the counts above. `median_hours` is lowest for `QUEENS` (about 4.6 hours) and highest for `BRONX` (about 23 hours). `open_or_unknown` counts requests with no usable duration.

Notice that `Borough` is now the **index** (the row labels), not a normal column.

The same table with the magnitudes made visible: in-cell bars for `complaints`, a colour gradient (darker = bigger) for the other columns. Below, one column as a sorted bar chart with the value written on each bar.

In [ ]:
styled_agg(by_borough)

In [ ]:
bar_labeled(by_borough['median_hours'], 'Median hours to close a request, by borough', 'Hours (median)', fmt='{:.1f} h')

**What to notice:** the Bronx stands out (highlighted) at roughly five times Queens, which the table alone makes you hunt for. Sorting plus value labels means no one has to read the axis.

### ✏️ Your turn

Compute the **median** resolution time (`hours`) per `Borough` and store it in `median_by_borough`. It should be a Series with one number per borough.

In [ ]:
median_by_borough = ...  # <- replace ... with your code

In [ ]:
check_equal(lambda: median_by_borough.round(2).to_dict(),
            {'BRONX': 23.07, 'BROOKLYN': 9.46, 'MANHATTAN': 7.28, 'QUEENS': 4.57, 'STATEN ISLAND': 11.85},
            hint="df.groupby('Borough')['hours'].median()")

<details>
<summary>Show solution</summary>

```python
median_by_borough = df.groupby('Borough')['hours'].median()
```

</details>

## 4. More than one key, and `reset_index()`

Give `groupby` a list to split by several columns at once. Here: borough *and* status.

In [ ]:
by_two = df.groupby(['Borough', 'Status']).size()
by_two.sort_values(ascending=False).head(5)

**You should see:** a Series whose index has two levels (`Borough` and `Status`). The largest group is `BROOKLYN` with status `Closed` (15967), then `BROOKLYN` `Open` (14224).

A result with a multi-level index is awkward to filter or plot. `reset_index()` moves the labels back into ordinary columns, and `name=` labels the counts:

In [ ]:
table = by_two.reset_index(name='complaints')
table.sort_values('complaints', ascending=False).head(5)

**You should see:** an ordinary table with three columns: `Borough`, `Status`, `complaints`, and plain row numbers on the left.

A two-key result is really a grid: boroughs down the side, statuses across the top. `unstack()` turns the second key into columns, and a heat-map paints each cell. Cells are shown as a percent of the borough's complaints, so big and small boroughs compare fairly.

In [ ]:
grid = pivot_heatmap(df, 'Borough', 'Status')

**What to notice:** every cell is one group from `groupby(['Borough', 'Status'])`. Each row adds up to 100%, so you can compare rows. The darkest cell in a row is that borough's most common status.

## 5. The easy way to get a share: the mean of a True/False column

In `10_Filter` you may have counted noise complaints per borough, then divided by all complaints per borough. There is a shortcut. In Python `True` is `1` and `False` is `0`, so the **mean of a boolean column is the share of `True`s**.

In [ ]:
df['is_noise'] = df['Complaint Type'].str.contains('Noise')

noise_share = df.groupby('Borough')['is_noise'].mean()
noise_share

**You should see:** five numbers between 0 and 1. `MANHATTAN` is highest at about `0.179` (nearly 18% of its complaints are about noise), and `BRONX` is lowest at about `0.037`.

Let's draw it, as percentages, sorted so the chart is easy to read:

In [ ]:
ax = (noise_share * 100).sort_values().plot(kind='barh', figsize=(7, 3))
ax.set_title('Share of complaints that are about noise')
ax.set_xlabel('Percent of the borough\'s complaints')
ax.set_ylabel('Borough')
plt.show()

**You should see:** five horizontal bars, with the shortest at the bottom (`BRONX`, about 3.7) and the longest at the top (`MANHATTAN`, about 17.9). Both axes have labels and the chart has a title.

### ✏️ Your turn

What share of each borough's complaints are about heating? Create a True/False column `is_heating` (complaint type exactly `'HEATING'`) and store the share per borough in `heating_share`.

In [ ]:
df['is_heating'] = ...      # <- replace ... with a True/False condition
heating_share = ...          # <- replace ... with your groupby

In [ ]:
check_equal(lambda: heating_share.round(4).to_dict(),
            {'BRONX': 0.2079, 'BROOKLYN': 0.1383, 'MANHATTAN': 0.1439, 'QUEENS': 0.0851, 'STATEN ISLAND': 0.0351},
            hint="df['Complaint Type'] == 'HEATING', then df.groupby('Borough')['is_heating'].mean()")

<details>
<summary>Show solution</summary>

```python
df['is_heating'] = df['Complaint Type'] == 'HEATING'
heating_share = df.groupby('Borough')['is_heating'].mean()
```

</details>

## 6. Bike counts: weekdays and months

Same idea, different dataset: how many cyclists crossed seven Montreal counters in 2012? The file uses `;` as separator and day-first dates (`01/01/2012`). Two columns are entirely empty (the counters had no data), so we drop them.

In [ ]:
bikes = pd.read_csv('bikes.csv', sep=';', encoding='latin1',
                    parse_dates=['Date'], dayfirst=True, index_col='Date')
bikes = bikes.dropna(axis=1, how='all')
print(bikes.shape)
bikes.head(3)

**You should see:** `(310, 7)` and a table with the dates as row labels and seven counter columns such as `Berri 1`, `Rachel1` and `Maisonneuve 2`. (Some names show a `?` because of accents. That is how the file was saved.)

The dates are the *index*, so we can group by something computed from it, such as `bikes.index.day_name()`. Let's look at one busy counter, `Berri 1`:

In [ ]:
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

per_weekday = bikes['Berri 1'].groupby(bikes.index.day_name()).agg(['sum', 'mean'])
per_weekday = per_weekday.reindex(order)   # put the days in calendar order
per_weekday

**You should see:** seven rows from `Monday` to `Sunday` with two columns. `Thursday` has the biggest total (160131), and `Sunday` the smallest (99310). Weekend days are clearly quieter, so this counter is mostly used for commuting.

In [ ]:
ax = per_weekday['mean'].plot(kind='bar', figsize=(7, 3.5))
ax.set_title('Berri 1: average cyclists per day, by weekday (2012)')
ax.set_xlabel('Weekday')
ax.set_ylabel('Average cyclists per day')
plt.show()

**You should see:** seven bars. Monday to Friday are tall (about 3000-3600), with Thursday the highest; Saturday and Sunday are noticeably shorter (about 2200-2300).

Now by month: `bikes.index.month` gives numbers 1-12.

In [ ]:
per_month = bikes['Berri 1'].groupby(bikes.index.month).sum()

ax = per_month.plot(kind='bar', figsize=(7, 3.5))
ax.set_title('Berri 1: total cyclists per month (2012)')
ax.set_xlabel('Month (1 = January)')
ax.set_ylabel('Total cyclists')
plt.show()

**You should see:** a bell shape: tiny bars in winter (months 1 and 2 only a few thousand), growing through spring to a peak in month 7 (July, about 162000), then falling again. The data stops in early November, so month 11 is small for that reason, not because of the weather.

### Seeing the bike patterns

The same groupings, drawn so the answer jumps out: the busiest weekday is highlighted and annotated, summer is shaded on the monthly line, and a weekday x month grid shows both keys at once.

In [ ]:
weekday_bars(bikes['Berri 1'])

**What to notice:** the orange bar is the busiest weekday; the grey band marks the weekend, where traffic drops. Commuters, not tourists, dominate.

In [ ]:
month_line(bikes['Berri 1'])

**What to notice:** the shaded summer months sit at the top of the curve. The last point falls because the data ends in early November.

In [ ]:
grid_bikes = weekday_month_heatmap(bikes['Berri 1'])

**What to notice:** read down a column to compare weekdays within a month, across a row to compare months. The dark band is summer weekdays; the pale weekend rows in cold months are the quietest groups of the year.

### ✏️ Your turn

For the counter `'Rachel1'`, compute the **total** cyclists per month and store it in `rachel_months`. Then store the number of the busiest month in `rachel_best_month`.

In [ ]:
rachel_months = ...      # <- replace ... with your code
rachel_best_month = ...  # <- replace ... with your code

In [ ]:
check_equal(lambda: int(rachel_months.loc[7]), 150703, hint="bikes['Rachel1'].groupby(bikes.index.month).sum()")
check_equal(lambda: int(rachel_best_month), 7, hint="Use .idxmax() on the monthly totals")

<details>
<summary>Show solution</summary>

```python
rachel_months = bikes['Rachel1'].groupby(bikes.index.month).sum()
rachel_best_month = rachel_months.idxmax()
```

</details>

## ✅ Check yourself

1. In your own words, what are the three steps of split-apply-combine?
2. What is the difference between `groupby('Borough').size()` and `groupby('Borough')['hours'].count()`?
3. Why does the mean of a True/False column give a percentage?
4. Why use `reset_index()` after grouping by two keys?
5. What does `groupby(bikes.index.month)` do when the month is not a column?

<details>
<summary>Show answers</summary>

1. Split the rows into groups, apply a calculation to each group, combine the results into one table.
2. `size()` counts all rows; `count()` counts only rows where `hours` is not missing.
3. `True` counts as 1 and `False` as 0, so the average is the fraction of `True`s.
4. It turns the group labels from a multi-level index back into ordinary columns, which are easier to sort, filter and plot.
5. You can pass any array of the same length as the data, such as one computed from the index. Rows with the same value land in the same group.

</details>

## Next

In `13_Mini_Project` you will combine everything from this module (loading, filtering, dates, grouping and plotting) to answer a question of your own.